[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02b/Demonstration.ipynb)


<!-- gen:values.yml#weeks.02b.notebook-title|Demonstration -->
# Week 2b Demonstration: Advanced Python
<!-- /gen -->

*Worked examples: Comprehensions, `lambda` with `sorted`, error handling, and a first class, each one an [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) draft you learn to read, judge, and direct.*

## How to work through this notebook

**Time:** about 60–75 minutes for the core sections, in your own time. The final 🧗 **Stretch** section is optional.

Every example is a draft your Associate wrote. For each one, use the three reading strategies from the [Introduction](./Introduction.ipynb). Trace the **data flow**, **name the <!-- plain-language: everyday intent -->intent** of each block, and **run fragments** in spare cells. The 🤖 **[AI Lens](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-lens)** boxes give you prompts to try on your own Associate (GitHub Copilot's chat panel). Do them.

The client is **Beans of Egham**, the Week 1 coffee cart. Its founder is asking about loyalty cards and quiet Mondays.

This cell builds one week of Beans of Egham transactions with a fixed seed (the same random numbers every run). One new piece of notation: `order_no += 1` means `order_no = order_no + 1`. It adds, then reassigns, in one step. Python calls this an [augmented assignment](https://docs.python.org/3/reference/simple_stmts.html#augmented-assignment-statements). `-=` and similar operators work the same way. Run the cell.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import numpy as np

rng = np.random.default_rng(5813)  # module seed: Do not change

# A full week of Beans of Egham transactions, one dict per sale.
MENU = {"flat white": 3.40, "latte": 3.60, "cappuccino": 3.50,
        "espresso": 2.20, "mocha": 3.80, "breakfast tea": 2.40}
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
drinks = list(MENU)

sales = []
order_no = 100
for day in DAYS:
    busy = 1.0
    if day in ("Friday", "Saturday"):   # weekend rush
        busy = 1.6
    for _ in range(int(rng.integers(18, 25) * busy)):
        order_no += 1
        drink = str(rng.choice(drinks, p=[0.24, 0.22, 0.16, 0.12, 0.14, 0.12]))
        sales.append({
            "order": order_no,
            "day": day,
            "drink": drink,
            "cups": int(rng.choice([1, 2, 3], p=[0.6, 0.3, 0.1])),
            "price_per_cup": MENU[drink],
            "loyalty_member": bool(rng.random() < 0.35),
        })

print(f"Created sales: {len(sales)} transactions across {len(DAYS)} days")
print(f"Example: {sales[0]}")

## Comprehensions: A list in one move

**The business ask:** *"Which orders were bought by loyalty members, and what did each spend?"*

Week 1's answer is a loop with `.append()`. The loop below collects each member order's spend, then prints the count and the total.

In [ ]:
member_spend = []
for sale in sales:
    if sale["loyalty_member"]:
        member_spend.append(sale["cups"] * sale["price_per_cup"])

print(f"{len(member_spend)} member orders, "
      f"total £{sum(member_spend):.2f}")

A **list comprehension** does the same in one line: Make a list, from a loop, with an optional filter. The Python tutorial's [list comprehensions](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions) section is the reference. Your Associate uses this shape constantly, so read it as three parts:

```python
[ expression  for item in collection  if condition ]
```

- `expression`: What to keep
- `for item in collection`: Where it comes from
- `if condition`: The filter, which is optional

Read aloud: *"The spend, for each sale, if the buyer is a member."*

In [ ]:
member_spend = [sale["cups"] * sale["price_per_cup"]
                for sale in sales
                if sale["loyalty_member"]]

print(f"{len(member_spend)} member orders, "
      f"total £{sum(member_spend):.2f}")

Same numbers. When you rewrite code, check that the outputs match. Here are two more shapes you will meet in Associate drafts. First, counting: Write the comprehension inside `sum()` with no square brackets. Each sale that passes the filter adds 1.

In [ ]:
# Counting with a comprehension: How many Saturday orders?
saturday_orders = sum(1 for sale in sales if sale["day"] == "Saturday")
print(f"Saturday orders: {saturday_orders}")

Second, a **dict comprehension** uses curly braces and builds `key: value` pairs. Here it builds next year's menu after a 10% price rise.

In [ ]:
# A DICT comprehension: Drink -> price including a 10% price rise
raised = {drink: round(price * 1.10, 2) for drink, price in MENU.items()}
print(raised)

### When a comprehension is too clever

Write a comprehension for the person who reads it next. If you cannot read it aloud as a sentence, it is too clever. Asked for "revenue per day, busiest first, as one expression", your Associate may produce this:

In [ ]:
# The Associate's 'one-liner'. It runs, and it is correct. Is it readable?
by_day = dict(sorted({d: sum(s["cups"] * s["price_per_cup"]
                             for s in sales if s["day"] == d)
                      for d in DAYS}.items(),
                     key=lambda kv: kv[1], reverse=True))
print(by_day)

Correct, and nearly unreadable. A comprehension inside a dict comprehension inside `sorted` inside `dict` is a trap. The next <!-- plain-language: everyday Analyst -->analyst (or you, in three weeks) has to take it apart before they can trust it or change it.

**The judgement rule:** One idea per comprehension. More than one, and you ask your Associate for the loop version.

> 🤖 **AI Lens:** Paste the one-liner above into your Associate with the first prompt below. Then ask it to explain the one-liner three different ways with the second prompt. Which explanation would you trust in front of a client, and which *version* would you ship?

```text
<paste the one-liner here>

Rewrite this as a plain loop with intermediate variables and one comment per step.
```

```text
<paste the one-liner here>

Explain this code three different ways: to a programmer, to a manager, and to a 10-year-old.
```

**A good answer** to the first prompt is a loop that prints the same `by_day` dictionary as the one-liner, with one idea per line. Ship the version whose explanation you would trust in front of a client.

## `lambda` and `sorted(key=...)`: Rank anything by anything

**The business ask:** *"What were our five biggest single sales this week?"*

`sorted()` can rank any collection. For a list of dicts, you tell it what to rank **by**. That is the `key` argument: A small function that takes one item and returns the value to sort on. The [Sorting HOW TO](https://docs.python.org/3/howto/sorting.html) is the official guide.

In [ ]:
def sale_value(sale):
    """What one transaction was worth, in pounds."""
    return sale["cups"] * sale["price_per_cup"]

top_five = sorted(sales, key=sale_value, reverse=True)[:5]

for sale in top_five:
    print(f"#{sale['order']} {sale['day']:9s} {sale['drink']:14s} "
          f"£{sale_value(sale):.2f}")

`key=` receives **the function itself** (`sale_value`), not a call to it (`sale_value(...)`). `sorted` calls it once per item.

When the key function is one expression used once, Python offers a shorthand: **`lambda`** ([lambda expressions](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions) in the Python tutorial), an unnamed function written inline. Your Associate prefers this form. Read it as: *"Given a sale, use cups × price."*

In [ ]:
top_five = sorted(sales,
                  key=lambda sale: sale["cups"] * sale["price_per_cup"],
                  reverse=True)[:5]

print(f"Biggest sale of the week: {top_five[0]}")

**Which to use?**

- `lambda`: Use it for a one-expression key that reads at a glance
- A named function (`def`): Use one when the logic needs a name, a second line, or reuse. `key=sale_value` says its <!-- plain-language: everyday intent -->intent. `key=lambda s: s["cups"] * s["price_per_cup"]` makes every reader work it out again

`min()` and `max()` accept the same `key=` argument.

> 🤖 **AI Lens:** Ask your Associate for the week's *quietest* day by revenue with the prompt below. Before you run its draft, predict the answer from the `by_day` output in "When a comprehension is too clever". [PREDICT-RUN](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-predict-run) works on AI output as it works on your own code.

```text
I have a Python list called `sales` where each item is a dict with keys `day`, `cups`, and `price_per_cup`, and a list `DAYS` of the six day names. Write one expression using min() with a key= argument that returns the day with the lowest total revenue, where a sale's revenue is cups times price_per_cup.
```

**A good answer** uses `min(DAYS, key=...)` with a key that totals `cups * price_per_cup` over each day's sales. The answer names the same day as the bottom of the `by_day` output.

## Error handling: Fail loudly, on purpose

**The business reality:** Data arrives broken. The card reader fails for a moment, a menu item is renamed, and a field goes missing. Code that only works on perfect data fails the first time a record is broken.

Here is a record the card machine mangled. The price was never recorded.

In [ ]:
# A mangled record from the card machine, and a crash. (This cell is EXPECTED to fail.)
mangled = {"order": 999, "day": "Saturday", "drink": "flat white", "cups": 2}

takings = mangled["cups"] * mangled["price_per_cup"]
print(takings)

A `KeyError` names the missing field. A traceback is the error report Python prints when code crashes. Read it from the bottom up. A crash is information: It tells you that something is wrong, where it went wrong, and what kind of problem it is.

`try`/`except` ([errors and exceptions](https://docs.python.org/3/tutorial/errors.html) in the Python tutorial) lets you plan for the failure instead of crashing on it. The shape:

```python
try:
    # the risky operation
except KeyError:
    # what to do when THAT specific thing goes wrong
```

The function below prices one sale and reports any record it skips.

In [ ]:
def order_value(sale):
    """Value of one sale in pounds, or None if the record can't be priced."""
    try:
        return sale["cups"] * sale["price_per_cup"]
    except KeyError as missing:
        print(f"⚠️ Order {sale.get('order', '?')} skipped: missing {missing}")
        return None

print(order_value(sales[0]))   # a healthy record
print(order_value(mangled))    # the broken one: Handled, and REPORTED

Two design choices in that function are the lesson:

1. **It catches `KeyError` specifically**, the failure it expects. A typo, a wrong type, or a new problem still crashes loudly, so you see it
2. **It reports what it skipped.** The <!-- plain-language: everyday Analyst -->analyst who runs it knows how many records fell out and why. A total that quietly leaves out broken records looks exactly like a correct total

### The anti-pattern your Associate loves

AI drafts often produce "defensive" code like this. Watch what it does to a typo:

In [ ]:
# The Associate's 'robust' version. It runs. Count the takings... carefully.
def order_value_robust(sale):
    try:
        return sale["cups"] * sale["price_per_cups"]   # <- typo: price_per_cupS
    except:              # bare except: Catches EVERYTHING
        return 0          # ... and pretends nothing happened

week_total = sum(order_value_robust(sale) for sale in sales)
print(f"Week's takings: £{week_total:.2f}")

**£0.00.** Every record "failed" because of a one-letter typo, and the bare `except:` turned each failure into a quiet zero. No error, no warning, a well-formatted wrong answer. In a real memo, that number goes to a founder deciding whether the cart survives the winter.

Here the damage is obvious because every record came out as zero. The dangerous version is when only some records fail: The total looks believable and nobody checks. You will meet exactly that in [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) 02b's [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags).

**House rules for `try`/`except`:**

- Catch the narrowest exception that names the failure you expect
- Never write a bare `except:` or `except: pass`. Silencing errors is not handling them
- Count and report whatever you skip. If 34 records fell out, the reader of your memo needs to know

> 🤖 **AI Lens:** Ask your Associate the first question below. Then ask it to *audit its own last draft* for the pattern, with the second prompt.

```text
Why is a bare `except` dangerous? Show me a case where it hides a bug that a specific `except KeyError` would have exposed.
```

```text
Audit your last draft for the pattern you just described: any bare `except`, or any `except` that silently swallows an error. For each one, name the specific exception it should catch instead and rewrite it.
```

**A good answer** shows a bare `except` turning a misspelled key into a quiet wrong result. With the same typo, `except KeyError` would have crashed and pointed at it. The audit then either finds the pattern in its own draft or shows there was none to find.

## A first class: Data and behaviour in one box

**The business ask:** The founder wants a loyalty scheme. *"A stamp per cup, a free drink every ten stamps."*

When data and the functions that act on it belong together, Python offers a **class**: A template that bundles them into one thing. The [Classes chapter](https://docs.python.org/3/tutorial/classes.html) of the Python tutorial covers more than this week needs. Read its first two sections.

From Week 4 you work in pandas, where every DataFrame and Series is an object: One thing built from a class. The class below is one customer's loyalty card.

In [ ]:
class LoyaltyCard:
    """One customer's stamps at Beans of Egham: a stamp per cup,
    a free drink every 10 stamps."""

    def __init__(self, owner):
        self.owner = owner
        self.stamps = 0
        self.free_drinks_earned = 0

    def add_stamps(self, cups):
        """Record a purchase of `cups` cups."""
        self.stamps += cups
        while self.stamps >= 10:
            self.stamps -= 10
            self.free_drinks_earned += 1

    def summary(self):
        """One line for the till screen."""
        return (f"{self.owner}: {self.stamps}/10 stamps, "
                f"{self.free_drinks_earned} free drink(s) earned")

### How to *read* a class (data flow first, always)

1. **Start at `__init__`**: It lists every piece of data the object carries (`owner`, `stamps`, `free_drinks_earned`). These are the class's "columns"
2. **Methods are verbs**: A method is a function that belongs to the class. `add_stamps` changes the data and `summary` reports it. Name each one's <!-- plain-language: everyday intent -->intent, as you would a block of a script
3. **`self` means "this particular card"**: Priya's card, not the idea of a card. Every card made from the template keeps its own data, as the next cell shows:

In [ ]:
priya = LoyaltyCard("Priya")
marco = LoyaltyCard("Marco")

priya.add_stamps(4)
priya.add_stamps(8)    # crosses 10 -> one free drink, 2 stamps carry over
marco.add_stamps(3)

print(priya.summary())
print(marco.summary())

You have used this notation all along: `"latte".upper()` and `prices.append(...)`. The pattern is value, dot, verb. Later, `df.groupby(...)` reads the same way. Strings, lists, and DataFrames are all objects.

**Judgement note:** A class is worth it when data and behaviour travel together and you need many independent copies. For a one-off calculation, a dict and a function are enough. Asked vaguely, an Associate may deliver a five-class design for a ten-line problem. Say no. That is [GENERATE-JUDGE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-generate-judge).

> 🤖 **AI Lens:** Ask your Associate to explain `self` three ways with the prompt below. Then test *your* reading: Predict what `priya.add_stamps(25)` leaves on her card, run the fragment, and compare. Also try `LoyaltyCard("Sam").summary()`. Reading the data flow, why does that work?

```text
Explain what `self` means in a Python class three ways: to a Java programmer, to an Excel user, and to a 10-year-old.
```

**A good answer** says the same thing in all three voices (`self` is this particular object, Priya's card and not the idea of a card). Then check that your prediction for `priya.add_stamps(25)` matches what the fragment prints.

## 🧗 Stretch (optional)

**Everything below is optional.** Nothing in Quest 02b or the assessments needs it. If you skip it, head to the [Quest](./Quest.ipynb).

### Stretch 1: `map` and `filter`, the functional ancestors

`map()` (apply a function to every item) and `filter()` (keep the items that pass a test) are older than comprehensions. Both are on the [built-in functions](https://docs.python.org/3/library/functions.html#map) page. You will meet them in older code and in some Associate drafts. The cell below compares both styles.

In [ ]:
values = [sale["cups"] * sale["price_per_cup"] for sale in sales[:6]]

# map/filter style:
with_vat = list(map(lambda v: round(v * 1.2, 2), values))
big = list(filter(lambda v: v > 5, values))

# comprehension style (same results, generally preferred today):
with_vat_c = [round(v * 1.2, 2) for v in values]
big_c = [v for v in values if v > 5]

print(with_vat == with_vat_c, big == big_c)

### Stretch 2: Sorting by more than one thing

A key function can return a **tuple**: A fixed sequence of values. Python compares the first value, then uses the second as a tie-break. Here that is day order, then biggest sale first. The minus sign reverses the order of the second value only. One new function: [`enumerate`](https://docs.python.org/3/library/functions.html#enumerate) numbers a collection as it loops (Monday 0, Tuesday 1, …). That turns day names into sortable positions.

In [ ]:
day_rank = {day: i for i, day in enumerate(DAYS)}   # Monday=0 ... Saturday=5

ordered = sorted(sales,
                 key=lambda s: (day_rank[s["day"]],
                                -s["cups"] * s["price_per_cup"]))

for sale in ordered[:3]:   # Monday's three biggest sales
    print(sale["day"], sale["drink"], sale["cups"], "cups")

### Stretch 3: Generator expressions, comprehensions without the list

Swap the square brackets for round ones and Python produces values one at a time instead of building the whole list in memory. You used this inside `sum(...)` when counting Saturday orders. For about 150 sales it changes nothing. For 150 million rows it is the difference between running and crashing.

In [ ]:
week_revenue = sum(s["cups"] * s["price_per_cup"] for s in sales)  # no [ ]
print(f"£{week_revenue:.2f}")

### Stretch 4: `__repr__`, and classes that cooperate

Most real classes have two more features. A [`__repr__`](https://docs.python.org/3/reference/datamodel.html#object.__repr__) method sets what the object shows when you print it. Python calls it for you. An object can also hold other objects.

In [ ]:
class Till:
    """The cart's till: knows every loyalty card it has issued."""

    def __init__(self):
        self.cards = {}

    def issue_card(self, owner):
        self.cards[owner] = LoyaltyCard(owner)
        return self.cards[owner]

    def __repr__(self):
        return f"Till({len(self.cards)} loyalty cards issued)"

till = Till()
till.issue_card("Priya").add_stamps(12)
till.issue_card("Marco")

print(till)                              # __repr__ at work
print(till.cards["Priya"].summary())     # objects inside objects

Read `till.cards["Priya"].summary()` slowly, tracing the data flow: A `Till` holds a dict, the dict holds a `LoyaltyCard`, the card answers `summary()`. Chains like this are what pandas code is made of. `df.groupby("day")["cups"].sum()` reads the same way in Week 4.

> 🤖 **AI Lens:** Ask your Associate to add a `spend_per_stamp` idea of its own to `LoyaltyCard`. Use the prompt below, and paste the class in place of the placeholder. Then judge the draft. Did it change `__init__`? Did it invent data the class never sets?

```text
<paste the LoyaltyCard class here>

Add a `spend_per_stamp` idea of your own to this class. Show the whole class after your change, and say what you added and why.
```

**A good answer** changes `__init__` only if the new idea needs new data, and uses only data the class actually sets. The answer also explains the addition in a sentence you can check against the code.

## Next

Head to [Quest.ipynb](./Quest.ipynb): **[Quest 02b](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest): The Subscription Churn Memo**. A streaming startup, a nervous board, and an Associate draft with something quietly wrong in it.